# RQ4: Scalability Evaluation
**Research Question 4:** *Is the proposed LSDM-structured RP framework computationally feasible for large-scale requirements prioritization and does it maintain acceptable execution times?*

To answer this, we evaluate the scalability of the framework from an empirical perspective. We apply the framework iteratively to various artificial datasets to explore both vertical scaling (increasing stakeholders) and horizontal scaling (increasing requirements).

**Experimental Design:**
* **Stakeholders (I):** {20, 50, 100, 200}
* **Requirements (M):** {100, 500, 1000}
* **Criteria (N):** Fixed at 5 (Importance, Benefit, Business Value, Risk, Implementation Duration)
* **Priority Classes (L):** Fixed at 4 (Must have, Should have, Could have, Won't have)
* **Sparsity:** Fixed at 60% missing ratings.
* **Neighborhood Parameters:** Dynamic, scaling at 10% of the stakeholder population.

We measure the memory consumption and the computational runtime of all five core modules of the framework, comparing the growth against our baseline configuration ($I=20$, $M=100$).

*(Note: The full grid results may be computationally intensive depending on the hardware. The results will vary for different hardware combinations. The full calculated results for the specific hardware of the experiment are already saved in the `/results` directory.)*

In [ ]:
# Import core libraries for time tracking, memory profiling, and data manipulation
import time
import tracemalloc
import pandas as pd
import warnings

# Import custom framework modules
from src.data_loader import generate_synthetic_data
from src.fuzzification import convert_to_ifv_matrix
from src.similarity import get_similarity_matrices
from src.recommender import predict_missing_values
from src.clustering import get_top_k_neighbors, calculate_snn_matrices, classify_stakeholders
from src.feedback import calculate_group_recommendation, calculate_final_classification

warnings.filterwarnings('ignore')

# Define parameters for vertical and horizontal scalability evaluation
stakeholder_sizes = [20, 50, 100, 200]
requirement_sizes = [100, 500, 1000]
n_criteria = 5
sparsity_level = 0.60

# Establish fixed optimal parameters for fuzzification and clustering based on prior evaluation
method = 'r-trapezoidal'
pi = 0.10
q_val = 0.15

results_scalability = []

print("Starting RQ4 Scalability Evaluation (12-Point Grid)...")
print("=" * 85)

for i_size in stakeholder_sizes:
    for m_size in requirement_sizes:
        print(f"Processing Configuration: I = {i_size:<4} | M = {m_size:<5} ...")

        # Dynamically scale neighborhood parameters relative to the stakeholder population
        pred_k = max(2, int(0.1 * i_size))
        clus_k = max(3, int(0.1 * i_size))

        # Compute total multi-dimensional data points for the current configuration
        data_points = i_size * m_size * n_criteria

        # Generate artificial multi-criteria dataset to simulate large-scale environments
        df_numeric, _, _, knowledge_matrix = generate_synthetic_data(
            n_stakeholders=i_size,
            n_reqs=m_size,
            n_cr=n_criteria,
            sparsity=sparsity_level,
            seed=100
        )
        stakeholders = df_numeric.index.tolist()

        # Start Memory Tracking
        tracemalloc.start()

        # Module 1: Intuitionistic Fuzzification
        t0 = time.time()
        df_ifv = convert_to_ifv_matrix(df_numeric, method=method, pi=pi)
        time_fuzz = time.time() - t0

        # Module 2: Pairwise Similarity Calculation (Computational Bottleneck)
        t0 = time.time()
        dist_matrix, sim_matrix = get_similarity_matrices(df_ifv)
        time_sim = time.time() - t0

        # Module 3: Collaborative Filtering Recommender System
        t0 = time.time()
        df_predicted = predict_missing_values(df_ifv, sim_matrix, k=pred_k, strategy='global')
        time_rs = time.time() - t0

        # Module 4: Shared Nearest Neighbor (SNN) Clustering
        t0 = time.time()
        neighbors_map = get_top_k_neighbors(sim_matrix, k=clus_k)
        snn_sim, snn_dist = calculate_snn_matrices(neighbors_map, stakeholders, k=clus_k)
        densities, core, isolated = classify_stakeholders(snn_sim, quantile=q_val)
        time_clust = time.time() - t0

        # Module 5: Feedback and Group Recommendation Synthesis
        t0 = time.time()
        df_final_ifv = df_ifv.copy()
        if len(core) > 0 and len(isolated) > 0:
            gr_series = calculate_group_recommendation(df_predicted, core)
            for st in isolated:
                df_final_ifv.loc[st] = gr_series
        time_gr = time.time() - t0

        # Module 6: Final Classification and Priority Ranking
        t0 = time.time()
        df_expert_ifv = convert_to_ifv_matrix(knowledge_matrix, method=method, pi=0.0)
        classification_df = calculate_final_classification(df_final_ifv, df_expert_ifv)
        time_rank = time.time() - t0

        # Stop Memory Tracking
        _, peak_memory = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        memory_mb = peak_memory / (1024 * 1024)

        total_time = time_fuzz + time_sim + time_rs + time_clust + time_gr + time_rank

        # Append to results
        results_scalability.append({
            'Stakeholders (I)': i_size,
            'Requirements (M)': m_size,
            'Total Data Points (I ⋅ M ⋅ N)': data_points,
            'Memory (MB)': memory_mb,
            'Fuzz (s)': time_fuzz,
            'Sim (s)': time_sim,
            'RS (s)': time_rs,
            'Clust (s)': time_clust,
            'GR Calc (s)': time_gr,
            'Rank (s)': time_rank,
            'Total Runtime (s)': total_time
        })

print("=" * 85)
print("Scalability Processing Complete.")

# Create DataFrame
df_scal = pd.DataFrame(results_scalability)

# Isolate the baseline configuration to compute performance scaling multipliers
baseline_data_points = df_scal.loc[0, 'Total Data Points (I ⋅ M ⋅ N)']
baseline_runtime = df_scal.loc[0, 'Total Runtime (s)']

# Insert comparative scaling multipliers against the baseline configuration
df_scal.insert(
    3, 'Total Data Points vs Baseline',
    (df_scal['Total Data Points (I ⋅ M ⋅ N)'] / baseline_data_points)
)
df_scal['Total Runtime vs Baseline'] = df_scal['Total Runtime (s)'] / baseline_runtime

# Save results after processing
# scalability_results_path = '../results/rq4_scalability_results.csv'
# df_scal.to_csv(scalability_results_path, index=False)
# print(f"Results successfully saved to '{scalability_results_path}'.")
display(df_scal)

### 2. Output and Performance Metrics Formatting
The computational results are loaded and formatted for analytical review. The output highlights the memory consumption and isolated module runtimes, alongside the scaling multipliers mapping the total runtime growth against the baseline configuration.

In [2]:
import pandas as pd
scalability_results_path = '../results/rq4_scalability_results.csv'
df_scal_loaded = pd.read_csv(scalability_results_path)

print("--- Scalability Analysis Results ---")
format_dict = {
    'Total Data Points (I ⋅ M ⋅ N)': '{:,}',
    'Total Data Points vs Baseline': '{:.1f} ×',
    'Memory (MB)': '{:.2f}',
    'Fuzz (s)': '{:.2f}',
    'Sim (s)': '{:.2f}',
    'RS (s)': '{:.2f}',
    'Clust (s)': '{:.2f}',
    'GR Calc (s)': '{:.2f}',
    'Rank (s)': '{:.2f}',
    'Total Runtime (s)': '{:.2f}',
    'Total Runtime vs Baseline': '{:.1f} ×'
}

# Display the final stylized table
display(df_scal_loaded.style.format(format_dict).hide(axis="index"))

--- Scalability Analysis Results ---


Stakeholders (I),Requirements (M),Total Data Points (I ⋅ M ⋅ N),Total Data Points vs Baseline,Memory (MB),Fuzz (s),Sim (s),RS (s),Clust (s),GR Calc (s),Rank (s),Total Runtime (s),Total Runtime vs Baseline
20,100,"10,000",1.0 ×,4.47,0.16,0.82,0.35,0.23,0.19,0.06,1.82,1.0 ×
20,500,"50,000",5.0 ×,20.30,0.86,3.16,1.63,0.21,0.75,0.27,6.88,3.8 ×
20,1000,"100,000",10.0 ×,40.49,1.58,6.18,3.28,0.21,1.54,0.53,13.33,7.3 ×
50,100,"25,000",2.5 ×,10.13,0.22,4.77,0.80,1.09,0.30,0.10,7.28,4.0 ×
50,500,"125,000",12.5 ×,50.06,1.15,20.45,3.77,1.11,1.47,0.49,28.44,15.6 ×
50,1000,"250,000",25.0 ×,100.03,2.15,39.45,7.53,1.09,2.82,0.96,54.00,29.7 ×
100,100,"50,000",5.0 ×,20.39,0.34,19.33,1.62,4.09,0.57,0.17,26.12,14.4 ×
100,500,"250,000",25.0 ×,99.80,1.64,84.77,7.24,4.08,2.69,0.86,101.27,55.7 ×
100,1000,"500,000",50.0 ×,199.55,3.31,170.13,14.67,4.14,5.23,1.68,199.17,109.6 ×
200,100,"100,000",10.0 ×,41.48,0.66,79.60,3.22,15.94,1.09,0.32,100.84,55.5 ×
